<b> Basis Q&A chain and Agent over a SQL database

In [18]:
import urllib.parse
from langchain_community.utilities  import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.sql import SQLDatabaseChain
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_classic.schema import HumanMessage, SystemMessage
from langchain_classic.chains import create_sql_query_chain
from dotenv import load_dotenv
load_dotenv()
import os

In [48]:
# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.getenv("GOOGLE_API_KEY"),temperature=0)

In [49]:
prompts = PromptTemplate.from_template(
    """You are a MySQL expert. Given an input question, create a syntactically correct MySQL query to run.
    Do NOT prefix the output with anything. Do NOT wrap the code in backticks or ```sql code blocks. 
    Output ONLY the raw SQL query statement and nothing else.

    Limit your results to a maximum of {top_k} rows unless specified otherwise.
    
    Database Table Info:
    {table_info}

    Question: {input}
    """
)

In [ ]:
host = '127.0.0.1'
port = '3306'
username = 'root'
raw_password = 'xxxxxx' 
password = urllib.parse.quote(raw_password)

database_schema = 'sql_store' 

mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"

# Connect LangChain to the store database
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=2)

# Verify it works by printing the tables inside 'store'
print("Tables found in store database:", db.get_usable_table_names())

Tables found in store database: ['customers', 'order_item_notes', 'order_items', 'order_statuses', 'orders', 'products', 'shippers']


In [50]:
chain = create_sql_query_chain(llm , db, prompt=prompts)

In [32]:
print(db.dialect)
print(db.get_usable_table_names())
db.run("SELECT count(*) FROM customers LIMIT 10;")

mysql
['customers', 'order_item_notes', 'order_items', 'order_statuses', 'orders', 'products', 'shippers']


'[(10,)]'

In [51]:
response = chain.invoke({"question":"How many customers are there?"})

In [52]:
db.run(response)

'[(10,)]'

In [55]:
response = chain.invoke({"question":"Which state are customers most from?"})

In [58]:
db.run(chain.invoke({"question":"Which state are customers  from?"}))

"[('MA',), ('VA',), ('CO',), ('FL',), ('TX',)]"

In [60]:
response = chain.invoke({"question":"What customers order what items. Show me all"})
print(response)
print(db.run(response))

SELECT
  c.first_name,
  c.last_name,
  p.name AS product_name,
  oi.quantity,
  oi.unit_price
FROM customers AS c
JOIN orders AS o
  ON c.customer_id = o.customer_id
JOIN order_items AS oi
  ON o.order_id = oi.order_id
JOIN products AS p
  ON oi.product_id = p.product_id;
[('Ines', 'Brushfield', 'Lettuce - Romaine, Heart', 7, Decimal('6.99')), ('Ines', 'Brushfield', 'Broom - Push', 7, Decimal('6.40')), ('Ines', 'Brushfield', 'Lettuce - Romaine, Heart', 7, Decimal('9.17')), ('Clemmie', 'Betchley', 'Pork - Bacon,back Peameal', 3, Decimal('9.89')), ('Clemmie', 'Betchley', 'Sauce - Ranch Dressing', 2, Decimal('6.94')), ('Clemmie', 'Betchley', 'Island Oasis - Raspberry', 2, Decimal('8.59')), ('Elka', 'Twiddell', 'Brocolinni - Gaylan, Chinese', 4, Decimal('3.74')), ('Elka', 'Twiddell', 'Foam Dinner Plate', 10, Decimal('6.01')), ('Elka', 'Twiddell', 'Longan', 9, Decimal('4.28')), ('Ilene', 'Dowson', 'Foam Dinner Plate', 2, Decimal('9.10')), ('Ilene', 'Dowson', 'Brocolinni - Gaylan, Chinese',